In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!unzip /content/drive/MyDrive/avocado.zip -d /content/dataset/

Archive:  /content/drive/MyDrive/avocado.zip
  inflating: /content/dataset/test/blackspot/b (1).jpg  
  inflating: /content/dataset/test/blackspot/b (10).jpg  
  inflating: /content/dataset/test/blackspot/b (11).jpg  
  inflating: /content/dataset/test/blackspot/b (12).jpg  
  inflating: /content/dataset/test/blackspot/b (13).jpg  
  inflating: /content/dataset/test/blackspot/b (14).jpg  
  inflating: /content/dataset/test/blackspot/b (15).jpg  
  inflating: /content/dataset/test/blackspot/b (16).jpg  
  inflating: /content/dataset/test/blackspot/b (17).jpg  
  inflating: /content/dataset/test/blackspot/b (18).jpg  
  inflating: /content/dataset/test/blackspot/b (19).jpg  
  inflating: /content/dataset/test/blackspot/b (2).jpg  
  inflating: /content/dataset/test/blackspot/b (20).jpg  
  inflating: /content/dataset/test/blackspot/b (21).jpg  
  inflating: /content/dataset/test/blackspot/b (22).jpg  
  inflating: /content/dataset/test/blackspot/b (3).jpg  
  inflating: /content/dataset/

In [5]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

print("Libraries imported successfully! TensorFlow version:", tf.__version__)

Libraries imported successfully! TensorFlow version: 2.21.0


In [7]:
train_dir = '/content/dataset/train'
test_dir = '/content/dataset/test'

# Image data generator with normalization and data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical'
)

Found 991 images belonging to 4 classes.
Found 99 images belonging to 4 classes.


In [8]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax') # 4 classes కాబట్టి 4 అخرాలు
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36992)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    18,940,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │         2,052 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,035,716 (72.62 MB)

 Trainable params: 19,035,716 (72.62 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    epochs=10,
    validation_data=test_generator,
    validation_steps=test_generator.samples // test_generator.batch_size
)

Epoch 1/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 30s 659ms/step - accuracy: 0.7372 - loss: 0.6931 - val_accuracy: 0.7812 - val_loss: 0.3493
Epoch 2/10
 1/30 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.8438 - loss: 0.2679

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.8438 - loss: 0.2679 - val_accuracy: 0.7500 - val_loss: 0.3792
Epoch 3/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 12s 396ms/step - accuracy: 0.8363 - loss: 0.2987 - val_accuracy: 0.8854 - val_loss: 0.2349
Epoch 4/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9062 - loss: 0.1768 - val_accuracy: 0.8750 - val_loss: 0.2411
Epoch 5/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 12s 393ms/step - accuracy: 0.8957 - loss: 0.2193 - val_accuracy: 0.9167 - val_loss: 0.1981
Epoch 6/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9062 - loss: 0.1760 - val_accuracy: 0.9167 - val_loss: 0.1970
Epoch 7/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 12s 389ms/step - accuracy: 0.9291 - loss: 0.1768 - val_accuracy: 0.9167 - val_loss: 0.1674
Epoch 8/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9375 - loss: 0.1748 - val_accuracy: 0.9271 - val_loss: 0.1351
Epoch 9/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 12s 384ms/step - accuracy: 0.9333 - loss: 0.1686 - val_accuracy: 0.8750 - val_

In [ ]:

test_loss, test_acc = model.evaluate(test_generator)
print(f"\nTest Accuracy: {test_acc * 100:.2f}%")

4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 540ms/step - accuracy: 0.8990 - loss: 0.1865

Test Accuracy: 89.90%


In [ ]:

model.save('greenpod_avocado_model.h5')
print("Model saved successfully as 'greenpod_avocado_model.h5'!")

Model saved successfully as 'greenpod_avocado_model.h5'!


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image


img_path = '/content/dataset/test/fresh/f (1).png'
img = image.load_img(img_path, target_size=(150, 150))
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0)
x /= 255.0

predictions = model.predict(x)
classes = list(train_generator.class_indices.keys())
predicted_class = classes[np.argmax(predictions)]

print(f"Predicted Avocado Condition: {predicted_class}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Predicted Avocado Condition: fresh
